# flacoAi — custom LoRA trainer (hackathon 3)

Trains a LoRA adapter on top of `Qwen/Qwen2.5-Coder-7B-Instruct` using Chris's rubric-conformant training pairs. Produces an adapter file + a GGUF export ready to serve via Ollama as `flaco-custom:7b`.

## What you need to do

1. **Runtime → Change runtime type → T4 GPU** (or A100 / L4 if you have Colab Pro).
2. Upload `train.merged.jsonl` to the session (left sidebar → Files → upload, or use the upload cell below).
3. **Runtime → Run all.**
4. When finished, download the `flaco-custom-lora.tar.gz` from the Files pane.
5. On the Mac, `tar -xzf flaco-custom-lora.tar.gz -C ~/infra/` and `cd ~/infra/flaco-custom && ollama create flaco-custom:7b -f Modelfile`.

Training a 7B LoRA on a T4 with ~25k pairs takes **~90-120 minutes**. If you are on the free tier the session will idle-disconnect after ~4 hours; we checkpoint every 100 steps so worst case you lose the tail of an epoch.

## 1. Install dependencies

Unsloth bundles a patched `transformers`, `peft`, `trl`, and `bitsandbytes` that trains 2-4x faster than the upstream stack. We use their free-tier-friendly loader.

In [ ]:
%%capture
!pip install --upgrade pip wheel
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets
!pip install sentencepiece protobuf

## 2. Upload the dataset

Run this cell and pick `train.merged.jsonl` from your local disk. Or drag the file onto the Colab Files pane and skip this cell.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('uploaded:', list(uploaded.keys()))

## 3. Load the base model with Unsloth

We use the 4-bit quantized `unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit` build for fast, low-VRAM training on a T4. Max sequence length 2048 — long enough for Chris's whole file rewrites and short enough to keep batches fast.

In [ ]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = 'unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit',
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,  # autodetect bf16/fp16
    load_in_4bit = True,
)

print('base model ready:', model.config._name_or_path)
print('cuda:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 4. Apply the LoRA config

Rank 32 is the sweet spot for style fine-tuning — high enough to capture Chris's architecture patterns, low enough to stay ~150 MB and train in under two hours on a T4. All attention + MLP projections are targeted so the adapter learns formatting, vocabulary, and reasoning adjustments uniformly.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

trainable, total = model.get_nb_trainable_parameters()
print(f'trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## 5. Load and format the dataset

The JSONL file has one `{"messages": [system, user, assistant]}` per line. We apply the Qwen chat template to turn each row into a single training string.

In [ ]:
from datasets import load_dataset

DATA_PATH = 'train.merged.jsonl'

raw = load_dataset('json', data_files=DATA_PATH, split='train')
print('rows:', len(raw))
print('first row keys:', raw[0].keys())

def format_row(row):
    return {
        'text': tokenizer.apply_chat_template(
            row['messages'],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

ds = raw.map(format_row, remove_columns=raw.column_names)
print('example formatted row (first 400 chars):')
print(ds[0]['text'][:400])

## 6. Train

3 epochs, effective batch size 8, lr 2e-4 with cosine decay. Checkpointing every 100 steps so disconnects are recoverable.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    dataset_text_field = 'text',
    max_seq_length = MAX_SEQ_LENGTH,
    packing = False,
    args = TrainingArguments(
        output_dir = 'flaco-custom-lora',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        lr_scheduler_type = 'cosine',
        warmup_ratio = 0.05,
        weight_decay = 0.01,
        optim = 'adamw_8bit',
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        save_strategy = 'steps',
        save_steps = 100,
        save_total_limit = 3,
        report_to = 'none',
        seed = 42,
    ),
)

stats = trainer.train()
print('done')
print(stats.metrics)

## 7. Quick smoke test

Sanity check one of the rubric-defining prompts against the trained model before exporting.

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = 'Write a SwiftUI view that owns a view model which needs the ExperimentationManager. Use Chris\'s preferred DI pattern.'

messages = [
    {'role': 'system', 'content': 'You are flacoAi, an expert in Chris Roura\'s Swift 6 / SwiftUI / POP architecture rubric.'},
    {'role': 'user', 'content': test_prompt},
]
input_ids = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors='pt'
).to('cuda')

out = model.generate(input_ids=input_ids, max_new_tokens=500, use_cache=True, temperature=0.2)
print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))

## 8. Export LoRA + Modelfile + tar up

We save the adapter, copy in the Modelfile for Ollama, and tar the whole thing for easy download.

In [ ]:
import os, shutil

EXPORT_DIR = 'flaco-custom'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Save LoRA adapter (safetensors) — this is what Ollama Modelfile will point at
model.save_pretrained(EXPORT_DIR)
tokenizer.save_pretrained(EXPORT_DIR)

# Write the Modelfile inside the export dir so it ships together
modelfile = '''FROM qwen2.5-coder:7b-instruct
ADAPTER ./adapter_model.safetensors
PARAMETER temperature 0.2
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
SYSTEM """You are flacoAi, a local-only AI brain for Chris Roura (Roura.io). You are an expert in Swift 6, SwiftUI, strict concurrency, and Chris's protocol-oriented architecture. You follow his rubric exactly: init DI for VM-backed views, @Entry + environment as fallback, actor wrappers for I/O, manager facades with .live/.mock factories, MARK-sectioned files, dedicated error enums. Never use @EnvironmentObject, singletons, @StateObject with parameterless init, or completion-handler APIs."""
'''
with open(os.path.join(EXPORT_DIR, 'Modelfile'), 'w') as f:
    f.write(modelfile)

# Tar for download
shutil.make_archive('flaco-custom-lora', 'gztar', '.', EXPORT_DIR)
print('wrote flaco-custom-lora.tar.gz')
!ls -la flaco-custom-lora.tar.gz
!du -sh flaco-custom/

In [ ]:
from google.colab import files
files.download('flaco-custom-lora.tar.gz')